# iSCORS — Classical Deliverable (γ + apparent α + density), no ML

The honest ACF-line output, computed by a **GPU classical fit in seconds — no network, no
training, no checkpoint** (the U-Net was retired; see `RETROSPECTIVE.md`).

- **γ map** — diffusion-rate map.
- **apparent global α** — one cell-wide anomalous exponent (~0.6); a *temporal-decorrelation*
  exponent (STICS could not verify it as true sub-diffusion — sub-PSF).
- **density G(0)=CV²** — the amplitude channel iSCORS normalises away: high-SNR, the
  cleanest single map (≈ condensation map), 'how much/many' complementary to γ's 'how fast'.

Paths/preprocessing mirror `iscors_real_runner.ipynb`. A Gradio front-end is at the end.


In [ ]:
# ── setup ────────────────────────────────────────────────────────────────
import os, subprocess, sys
REPO='https://github.com/breezy90126/iscors-net.git'; BRANCH='claude/brave-ramanujan-33eps3'
REPO_DIR='/content/iscors-net'
try:
    from google.colab import drive; drive.mount('/content/drive', force_remount=False)
except Exception: pass
if os.path.isdir(REPO_DIR):
    for c in (['git','-C',REPO_DIR,'fetch','origin'],['git','-C',REPO_DIR,'checkout',BRANCH],
              ['git','-C',REPO_DIR,'pull','origin',BRANCH]): subprocess.run(c, check=False)
else:
    subprocess.run(['git','clone','--branch',BRANCH,REPO,REPO_DIR], check=False)
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
subprocess.run(['pip','install','-q','tifffile','scipy','gradio'], check=False)
print('setup done:', os.getcwd())


In [ ]:
# ── config (same paths as iscors_real_runner.ipynb) ──────────────────────
import numpy as np
ZIP_PATH    = '/content/drive/MyDrive/iscors_test/large_file.zip'   # .zip or .tif
EXTRACT_DIR = '/content/real_data'                                  # zip extraction cache
VIDEO_FNAME = 'COBRI_rarw_video.tif'                                # name inside the zip
MAT_FNAME   = 'Output_iSCORS_map.mat'                               # iSCORS GT (for Cond_map check)
N_FRAMES    = 2000
BIN_FACTOR  = 2
RECON_TAUS  = (1, 2, 4, 8, 16, 32, 48, 64, 96, 128)
GAMMA_SCALE = 2.0
CONDENSATION_SLOPE = 3.0      # log-log baseline slope (iSCORS condensation projection)
DSTAR_FROM = 'gamma'          # 1/D*: 'gamma' → 1/γ ; 'tauD' → τ_D = γ^(-1/α)
print('config ready')


In [ ]:
# ── reusable pipeline: load → preprocess → classical analyze ─────────────
import os, sys, numpy as np, tifffile, zipfile, tempfile
from scipy.ndimage import gaussian_filter
# robust path: don't rely on the setup cell's sys.path persisting
REPO_DIR = '/content/iscors-net'
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
import importlib, utils.gpu_iscors_fit as _gf; importlib.reload(_gf)
from utils.gpu_iscors_fit import gpu_fit_maps, compute_density, condensation, condensation_projection

def load_video(path, fname=None, n_frames=2000, bin_factor=2):
    if str(path).lower().endswith('.zip'):
        ed = globals().get('EXTRACT_DIR', '/content/real_data'); os.makedirs(ed, exist_ok=True)
        def _find(root, name):
            for dp, _, fs in os.walk(root):
                if name in fs: return os.path.join(dp, name)
            return None
        vp = _find(ed, fname)            # reuse cached extraction (no re-extract per call)
        if not vp:
            with zipfile.ZipFile(path) as z: z.extractall(ed)
            vp = _find(ed, fname)
        assert vp, f'{fname} not found in zip'
    else:
        vp = path
    H0, W0 = tifffile.imread(vp, key=0).shape
    Hb, Wb = (H0//bin_factor)*1, (W0//bin_factor)*1
    with tifffile.TiffFile(vp) as tf:
        try:    total = int(tf.series[0].shape[0])   # robust for large/BigTIFF
        except Exception: total = len(tf.pages)
    n = min(n_frames, total)
    # crop to a bin-divisible size (avoids reshape misalignment)
    Hc, Wc = Hb*bin_factor, Wb*bin_factor
    raw = np.empty((n, Hb, Wb), np.float32)
    for s in range(0, n, 100):
        e = min(s+100, n); ch = tifffile.imread(vp, key=range(s, e)).astype(np.float32)
        raw[s:e] = ch[:, :Hc, :Wc].reshape(e-s, Hb, bin_factor, Wb, bin_factor).mean((2, 4))
    # ── sanity: a >4GB classic TIFF (offset overflow) or truncated extraction
    #    yields garbage/constant frames → CV≈0 → 'no cell pixels'. Surface it here.
    finite = np.isfinite(raw).all()
    mean_I = raw.mean(0); cvm = raw.std(0) / (np.abs(mean_I) + 1e-10)
    ncell = int((cvm >= 0.005).sum())
    print(f'[load] {os.path.basename(vp)}: {n}/{total} frames  raw {raw.shape}  '
          f'mean={raw.mean():.3g} std={raw.std():.3g} min={raw.min():.3g} max={raw.max():.3g}')
    print(f'[load] cell pixels @CV>=0.005: {ncell}/{Hb*Wb}  ({100*ncell/(Hb*Wb):.1f}%)')
    if (not finite) or raw.std() < 1e-9 or ncell == 0:
        print('[load] *** DEGENERATE DATA *** — almost certainly a TIFF read problem:')
        print('       - >4GB classic TIFF: page offsets overflow at ~4GB (your warning was'
              ' offset≈4.10GB) → frames near/after 4GB read as garbage. Lower N_FRAMES so the'
              ' read stays under 4GB, or re-save the video as BigTIFF.')
        print('       - truncated extraction (disk full): re-extract, check `df -h /content`.')
        print('       Compare with iscors_real_runner.ipynb (same loader) to confirm.')
    return raw

def preprocess(raw):
    ff = raw / (np.median(raw, axis=0)[None] + 1e-10)           # flat-field
    proc = np.empty_like(ff)
    for t in range(len(ff)):
        proc[t] = ff[t] / (gaussian_filter(ff[t], sigma=4) + 1e-10)  # per-frame BG removal
    return proc

def analyze(video_proc):
    out = gpu_fit_maps(video_proc, recon_taus=RECON_TAUS, n_components=1, global_alpha=True,
                       gamma_scale=GAMMA_SCALE, min_cv=0.005, n_steps=500, verbose=False)
    cell = out['cell_mask']
    dens, _ = compute_density(video_proc, min_cv=0.005)             # V_DLS = CV²
    g = np.clip(out['gamma'], 1e-6, None)
    if DSTAR_FROM == 'tauD':
        inv_Dstar = np.power(g, -1.0/np.clip(out['alpha'], 0.1, 2.0))  # τ_D = γ^(-1/α)
    else:
        inv_Dstar = 1.0 / g                                          # 1/D* ∝ 1/γ
    # iSCORS condensation: slope-3 log-log perpendicular projection (V_DLS vs 1/D*)
    cond, cinfo = condensation_projection(dens, inv_Dstar, cell, slope=CONDENSATION_SLOPE)
    return dict(gamma=out['gamma'], alpha_global=float(np.nanmedian(out['alpha'][cell])),
                density=dens, inv_Dstar=inv_Dstar, condensation=cond, cell=cell,
                density_b=cinfo['b'])

def make_fig(data, cmap, title):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5, 4.5))
    p1, p99 = np.nanpercentile(data, 1), np.nanpercentile(data, 99)
    im = ax.imshow(data, cmap=cmap, vmin=p1, vmax=p99); plt.colorbar(im, ax=ax)
    ax.set_title(title, fontsize=11); ax.axis('off'); fig.tight_layout(); return fig
print('pipeline defined')


In [ ]:
# ── run on the configured file + plot ────────────────────────────────────
import matplotlib.pyplot as plt, time
t0 = time.time()
raw = load_video(ZIP_PATH, VIDEO_FNAME, N_FRAMES, BIN_FACTOR)
video_proc = preprocess(raw)
res = analyze(video_proc)
print(f'done in {time.time()-t0:.1f}s   apparent global α = {res["alpha_global"]:.3f}   '
      f'cell={100*res["cell"].mean():.1f}%   density intercept b = {res["density_b"]:.3f}')
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for a_, d, cm, ttl in [(ax[0], res['gamma'], 'magma', 'γ (diffusion rate)'),
                       (ax[1], res['density'], 'viridis', 'density  G(0)=CV²'),
                       (ax[2], res['condensation'], 'inferno', 'Condensation (slope-3 proj)')]:
    p1, p99 = np.nanpercentile(d, 1), np.nanpercentile(d, 99)
    im = a_.imshow(d, cmap=cm, vmin=p1, vmax=p99); plt.colorbar(im, ax=a_)
    a_.set_title(ttl, fontsize=11); a_.axis('off')
plt.suptitle(f'iSCORS classical deliverable — apparent global α ≈ {res["alpha_global"]:.2f}', fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# ── validate: our Condensation vs the .mat Cond_map (Pearson) ────────────
# Also pits raw CV², raw γ, and both Φ forms against Cond_map → tells which quantity
# the iSCORS condensation map actually is, and whether the CV²/Φ correction helps.
import os, numpy as np, matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from scipy.ndimage import zoom
from utils.gpu_iscors_fit import condensation

def _find(root, name):
    for dp, _, fs in os.walk(root):
        if name in fs: return os.path.join(dp, name)
    return None
mat_path = _find(EXTRACT_DIR, MAT_FNAME)
if not mat_path and str(ZIP_PATH).lower().endswith('.zip'):
    import zipfile
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(EXTRACT_DIR)
    mat_path = _find(EXTRACT_DIR, MAT_FNAME)
assert mat_path, f'{MAT_FNAME} not found under {EXTRACT_DIR}'
try:
    import scipy.io; mat = scipy.io.loadmat(mat_path); mat_keys=[k for k in mat if not k.startswith('_')]
except NotImplementedError:
    import h5py; mat={}
    with h5py.File(mat_path,'r') as hf:
        for k in hf: mat[k]=np.array(hf[k])
    mat_keys=list(mat)
assert 'Cond_map' in mat, f'Cond_map not in .mat; keys={mat_keys}'
cond_raw = np.asarray(mat['Cond_map']).astype(np.float64).squeeze()

cell = res['cell']; Hm, Wm = res['condensation'].shape
Hc, Wc = cond_raw.shape
if abs(Hc-Wm) < abs(Hc-Hm) and Hc != Hm: cond_raw = cond_raw.T   # transpose if column-major
def _align(a):
    return (a if (a.shape[0],a.shape[1])==(Hm,Wm)
            else zoom(a,(Hm/a.shape[0],Wm/a.shape[1]),order=1)).astype(np.float32)
def _coord(a):                                   # MATLAB→Python: flipud + rot90 clockwise
    o = np.rot90(np.flipud(a), k=-1).astype(np.float32)
    return o if o.shape==(Hm,Wm) else zoom(o,(Hm/o.shape[0],Wm/o.shape[1]),order=1).astype(np.float32)
cond_gt = _coord(_align(cond_raw)); cond_gt[~cell] = np.nan

# candidate maps to compare against Cond_map
a_full = np.full_like(res['gamma'], res['alpha_global'])
cands = {
    'Condensation (slope-3 proj)': res['condensation'],
    'CV²/γ (simple V_DLS/D)'     : condensation(res['density'], res['gamma'], a_full, blur='gamma'),
    'raw CV² (V_DLS)'            : res['density'],
    '1/γ (1/D*)'                : 1.0/np.clip(res['gamma'], 1e-6, None),
}
print('=== vs .mat Cond_map (cell pixels) ===')
best, best_p = None, -1
for name, m in cands.items():
    ok = cell & np.isfinite(cond_gt) & np.isfinite(m)
    x, y = cond_gt[ok].astype(float), m[ok].astype(float)
    pr = pearsonr(x, y)[0]; sr = spearmanr(x, y).statistic
    print(f'  {name:24s}  Pearson={pr:+.3f}  Spearman={sr:+.3f}  N={ok.sum()}')
    if abs(pr) > best_p: best, best_p = name, abs(pr)
print(f'  → best match to Cond_map: {best} (|Pearson|={best_p:.3f})')

# maps + scatter of the (default) condensation
our = cands['Condensation (slope-3 proj)']
ok = cell & np.isfinite(cond_gt) & np.isfinite(our)
pr = pearsonr(cond_gt[ok], our[ok])[0]; sr = spearmanr(cond_gt[ok], our[ok]).statistic
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
for a_, d, t in [(ax[0], cond_gt, '.mat Cond_map'),
                 (ax[1], our, 'our Condensation (slope-3 proj)')]:
    p1, p99 = np.nanpercentile(d, 1), np.nanpercentile(d, 99)
    im = a_.imshow(d, cmap='inferno', vmin=p1, vmax=p99); plt.colorbar(im, ax=a_, fraction=0.046)
    a_.set_title(t, fontsize=11); a_.axis('off')
ax[2].scatter(cond_gt[ok], our[ok], alpha=0.05, s=1, c='steelblue')
ax[2].set_xlabel('.mat Cond_map'); ax[2].set_ylabel('our Condensation')
ax[2].set_title(f'Pearson={pr:+.3f}  Spearman={sr:+.3f}')
fig.suptitle('Condensation validation vs iSCORS .mat Cond_map', fontsize=13)
plt.tight_layout(); plt.show()
if 'SAVE_DIR' in globals():
    fig.savefig(os.path.join(SAVE_DIR, 'condensation_vs_condmap.png'), dpi=120, bbox_inches='tight')


In [ ]:
# ── Stage A: classical noise-free V_DLS (amplitude A) vs raw CV²=G(0) ─────
# Raw V_DLS=G(0) includes the τ=0 noise spike; A extrapolates the noise-free τ≥1 ACF
# to τ→0. Does removing that noise bias improve the condensation (vs Cond_map)? If A≈raw
# the bias is small (no room for FAST); if A clearly beats raw, ML frame-denoising (Stage B)
# is worth trying. Needs cond_gt from the Cond_map validation cell above.
import numpy as np, matplotlib.pyplot as plt, os
from scipy.stats import pearsonr, spearmanr
from utils.gpu_iscors_fit import vdls_amplitude, condensation_projection

cell = res['cell']
a_full = np.full_like(res['gamma'], res['alpha_global'])
A   = vdls_amplitude(video_proc, RECON_TAUS, res['gamma'], a_full)   # noise-free V_DLS
cv2 = res['density']                                                 # raw V_DLS = G(0)
ratio = cv2 / np.clip(A, 1e-12, None)
print(f'V_DLS  raw/clean ratio (median over cell) = {np.nanmedian(ratio[cell]):.2f}  '
      f'(>1 ⇒ G(0) inflated by τ=0 noise; ≈1 ⇒ little noise to remove)')

inv_D = 1.0 / np.clip(res['gamma'], 1e-6, None)
cond_clean, _ = condensation_projection(A,   inv_D, cell, slope=CONDENSATION_SLOPE)
cond_raw      = res['condensation']                                  # from raw CV²

if 'cond_gt' in globals():
    print('\n=== vs .mat Cond_map (does noise-removal help?) ===')
    for nm, m in [('condensation (raw CV²)', cond_raw),
                  ('condensation (clean A)', cond_clean),
                  ('raw CV² (V_DLS)',        cv2),
                  ('clean A (V_DLS)',        A)]:
        ok = cell & np.isfinite(cond_gt) & np.isfinite(m)
        pr = pearsonr(cond_gt[ok], m[ok])[0]; sr = spearmanr(cond_gt[ok], m[ok]).statistic
        print(f'  {nm:24s}  Pearson={pr:+.3f}  Spearman={sr:+.3f}')
    print('  VERDICT: clean ≫ raw ⇒ noise bias matters → Stage B (FAST) worth it;'
          ' clean ≈ raw ⇒ V_DLS already clean, ML not needed.')
else:
    print('(run the Cond_map validation cell first to define cond_gt for comparison)')

fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
for a_, d, t in [(ax[0], cv2,        'raw V_DLS = CV² = G(0)'),
                 (ax[1], A,          'clean V_DLS = A (ACF τ→0)'),
                 (ax[2], cond_clean, 'Condensation (clean V_DLS)')]:
    p1, p99 = np.nanpercentile(d, 1), np.nanpercentile(d, 99)
    im = a_.imshow(d, cmap='inferno', vmin=p1, vmax=p99); plt.colorbar(im, ax=a_, fraction=0.046)
    a_.set_title(t, fontsize=11); a_.axis('off')
fig.suptitle('Stage A — classical noise-free V_DLS (amplitude extrapolation)', fontsize=13)
plt.tight_layout(); plt.show()
if 'SAVE_DIR' in globals():
    fig.savefig(os.path.join(SAVE_DIR, 'stageA_vdls_clean.png'), dpi=120, bbox_inches='tight')


In [ ]:
# ── slope diagnostic: is slope-3 real, shared-noise, or species-specific? ─
# V_DLS and 1/D* both come from the SAME per-pixel time series → their estimation NOISE
# is correlated and biases the empirical slope (noise: CV²↑, γ↑ ⇒ X↓,Y↑ ⇒ slope pulled
# DOWN, away from 3). Test: compare the cloud slope (PCA/TLS) from SHARED data vs from
# INDEPENDENT frame halves (V_DLS from first half, γ from second half → decorrelated noise,
# same physics). If the slope shifts → it was shared noise; if stable → real.
import os, numpy as np, matplotlib.pyplot as plt
from utils.gpu_iscors_fit import gpu_fit_maps, compute_density

def _tls_slope(vdls, gamma, cell):
    X = np.log10(np.clip(1.0/np.clip(gamma,1e-6,None), 1e-12, None))   # log(1/D*) = -log γ
    Y = np.log10(np.clip(vdls, 1e-12, None))                          # log(V_DLS)
    m = cell & np.isfinite(X) & np.isfinite(Y)
    x = X[m]-X[m].mean(); y = Y[m]-Y[m].mean()
    w, V = np.linalg.eigh(np.cov(np.vstack([x, y])))                  # ascending eigvals
    pc1 = V[:, 1]
    return float(pc1[1]/(pc1[0]+1e-12)), float(w[1]/(w.sum()+1e-12)), X, Y, m

s_sh, ve_sh, X, Y, m = _tls_slope(res['density'], res['gamma'], res['cell'])
T = video_proc.shape[0]; h = T//2
dens1, _ = compute_density(video_proc[:h], min_cv=0.005)              # V_DLS: first half
fit2 = gpu_fit_maps(video_proc[h:], recon_taus=RECON_TAUS, n_components=1, global_alpha=True,
                    gamma_scale=GAMMA_SCALE, min_cv=0.005, n_steps=400, verbose=False)  # γ: second half
celli = res['cell'] & np.isfinite(dens1) & np.isfinite(fit2['gamma'])
s_in, ve_in, *_ = _tls_slope(dens1, fit2['gamma'], celli)

print(f'empirical slope (shared data)        = {s_sh:+.2f}   PC1 variance {100*ve_sh:.0f}%')
print(f'empirical slope (independent halves) = {s_in:+.2f}   PC1 variance {100*ve_in:.0f}%')
print(f'physical baseline                    = 3.00')
print('Interpret:')
print(f'  • |Δslope| = {abs(s_in-s_sh):.2f}: large ⇒ shared estimation noise was biasing it')
print('  • ≈3 & high PC1 var ⇒ slope-3 physics holds')
print('  • ≠3 but stable across the split ⇒ genuine species/condition slope → use it, not 3')
print('  • low PC1 var ⇒ fat cloud, slope poorly defined (noise-dominated)')

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X[m], Y[m], s=1, alpha=0.05, c='steelblue')
xx = np.linspace(np.nanpercentile(X[m],1), np.nanpercentile(X[m],99), 50)
b3 = np.mean(Y[m]-3*X[m]); bs = np.mean(Y[m]-s_sh*X[m])
ax.plot(xx, 3*xx+b3, 'r-',  label='slope 3 (physics prior)')
ax.plot(xx, s_sh*xx+bs, 'g--', label=f'empirical PC1 (slope {s_sh:.2f})')
ax.set_xlabel('log10(1/D*) = -log10(γ)'); ax.set_ylabel('log10(V_DLS = CV²)')
ax.legend(fontsize=8); ax.set_title('Condensation feature cloud: slope-3 vs empirical')
plt.tight_layout(); plt.show()
if 'SAVE_DIR' in globals(): fig.savefig(os.path.join(SAVE_DIR,'slope_diagnostic.png'),dpi=120,bbox_inches='tight')


In [ ]:
# ── load hand-drawn nucleus mask → PCA / whitened / slope-3 projection ────
# Uses the uploaded mask file directly (no Otsu/GMM/auto-segmentation).
# (1) Load the mask, take its central region as the chromatin (nucleus) pixels.
# (2) PCA on the log(1/D*)–log(V_DLS) cloud of those pixels.
# (3) Compare three condensation baselines — slope-3 (physics), raw-PCA PC1,
#     whitened-PCA PC1 — reporting each one's slope, y-intercept (= overall mass
#     density in the iSCORS model) and Pearson vs the .mat Cond_map.
import os, numpy as np, matplotlib.pyplot as plt, tifffile
from scipy.ndimage import zoom

# ---------- 1. load the hand-drawn mask (no segmentation algorithm) -----------
MASK_PATH = 'data/condensation_mask.tif'      # the uploaded mask
mask_img = np.asarray(tifffile.imread(MASK_PATH)).squeeze()
if mask_img.ndim == 3:                         # RGB/indexed → collapse to a single channel
    mask_img = mask_img[..., 0] if mask_img.shape[-1] <= 4 else mask_img[0]
# resize (nearest) to the map grid if the mask was drawn at a different resolution
H, W = res['density'].shape
if mask_img.shape != (H, W):
    mask_img = zoom(mask_img, (H / mask_img.shape[0], W / mask_img.shape[1]), order=0)
# the mask has two levels; the nucleus is the INTERIOR region (it does not touch the
# frame edge), while the background wraps around it and fills the border. Pick the level
# that appears LEAST on the image border as the nucleus (robust to ImageJ's 0/255 polarity).
levels = np.unique(mask_img)
border = np.zeros(mask_img.shape, bool)
border[0, :] = border[-1, :] = border[:, 0] = border[:, -1] = True
fg_level = min(levels, key=lambda v: (mask_img[border] == v).mean())
nuc = (mask_img == fg_level)
print(f'[mask] {MASK_PATH}: levels={levels.tolist()}, nucleus level={fg_level} '
      f'(border-fraction {(mask_img[border]==fg_level).mean():.2f}), '
      f'nucleus pixels={nuc.sum()} ({100*nuc.sum()/nuc.size:.0f}% of frame)')

# ---------- 2. PCA on the log-log cloud of chromatin pixels --------------------
X = np.log10(np.clip(res['inv_Dstar'], 1e-12, None))     # log10(1/D*)  (slowness)
Y = np.log10(np.clip(res['density'],   1e-12, None))     # log10(V_DLS = CV²)
m = nuc & np.isfinite(X) & np.isfinite(Y)
print(f'[mask] valid (finite γ & V_DLS) nucleus pixels = {m.sum()}')
xc, yc = X[m].mean(), Y[m].mean()
xz, yz = X[m] - xc, Y[m] - yc

def _pc1_slope(u, v):                                     # PC1 direction of a 2D cloud
    w, V = np.linalg.eigh(np.cov(np.vstack([u, v])))      # eigvals ascending → take last
    pc1 = V[:, 1]
    return float(pc1[1] / (pc1[0] + 1e-12)), float(w[1] / (w.sum() + 1e-12))

s_pca,  ve_pca   = _pc1_slope(xz, yz)                     # raw-covariance PC1
sx, sy = xz.std(), yz.std()
s_w_std, ve_white = _pc1_slope(xz / sx, yz / sy)         # PC1 of the z-scored (whitened) cloud
s_white = s_w_std * (sy / sx)                            # map the slope back to original units

# ---------- 3. three baselines → slope, y-intercept, condensation map ----------
def project(slope, b):
    """Decompose each pixel along the baseline Y = slope·X + b:
      condensation = perpendicular-foot X-coord (position ALONG the line);
      density_map  = per-pixel y-intercept Y − slope·X (offset PERPENDICULAR to it,
                     = the iSCORS mass-density channel; global b = its mean)."""
    Xp = (X + slope * Y - slope * b) / (1.0 + slope ** 2)
    cond = Xp.astype(np.float32).copy(); cond[~nuc] = np.nan
    dens = (Y - slope * X).astype(np.float32).copy(); dens[~nuc] = np.nan
    return cond, dens

methods = {}                                            # name -> (slope, b, cond_map, density_map)
b3 = float((Y[m] - 3.0 * X[m]).mean())                  # slope-3 LS intercept = density
methods['slope-3 (physics)'] = (3.0,     b3,                       *project(3.0, b3))
methods['PCA (raw cov)']     = (s_pca,   float(yc - s_pca * xc),   *project(s_pca,   yc - s_pca * xc))
methods['PCA (whitened)']    = (s_white, float(yc - s_white * xc), *project(s_white, yc - s_white * xc))

print('\n=== baseline comparison (nucleus pixels) ===')
print(f'{"method":18s} {"slope":>7s} {"y-intercept(b)":>15s}')
for name, (s, b, _, _) in methods.items():
    print(f'{name:18s} {s:>7.2f} {b:>15.3f}')
print(f'(PC1 variance fraction: raw {100*ve_pca:.0f}%, whitened {100*ve_white:.0f}%)')

# ---------- optional: validate each baseline against the .mat Cond_map ---------
from scipy.stats import pearsonr
if 'cond_gt' in globals():
    print('\n=== condensation vs .mat Cond_map (nucleus pixels) ===')
    for name, (s, b, cond, dens) in methods.items():
        ok = nuc & np.isfinite(cond_gt) & np.isfinite(cond)
        if ok.sum() >= 2:
            print(f'  {name:18s} Pearson={pearsonr(cond_gt[ok], cond[ok])[0]:+.3f}  N={ok.sum()}')
        else:
            print(f'  {name:18s} skipped — only {ok.sum()} overlapping pixel(s)')
# the y-intercept (density) map should track the raw amount-of-chromatin channel (CV²)
print('\n=== density (y-intercept) map vs raw V_DLS=CV² (nucleus pixels) ===')
cv2log = Y                                               # log10(CV²) = the amount channel
for name, (s, b, cond, dens) in methods.items():
    ok = nuc & np.isfinite(dens) & np.isfinite(cv2log)
    print(f'  {name:18s} Pearson={pearsonr(cv2log[ok], dens[ok])[0]:+.3f}')

# ---------- plots -------------------------------------------------------------
fig = plt.figure(figsize=(16, 8))
ax = fig.add_subplot(2, 3, 1)
ax.imshow(res['density'], cmap='gray',
          vmin=np.nanpercentile(res['density'], 1), vmax=np.nanpercentile(res['density'], 99))
ax.contour(nuc, colors='r', linewidths=0.8)
ax.set_title('loaded nucleus mask on V_DLS'); ax.axis('off')
ax = fig.add_subplot(2, 3, 2)
ax.scatter(X[m], Y[m], s=1, alpha=0.05, c='steelblue')
xx = np.linspace(np.percentile(X[m], 1), np.percentile(X[m], 99), 50)
for (name, (s, b, _, _)), c in zip(methods.items(), ['r', 'g', 'm']):
    ax.plot(xx, s * xx + b, c, lw=1.5, label=f'{name}: slope {s:.2f}')
ax.set_xlabel('log10(1/D*)'); ax.set_ylabel('log10(V_DLS)')
ax.legend(fontsize=7); ax.set_title('cloud + three baselines')
for j, (name, (s, b, cond, dens)) in enumerate(methods.items()):
    ax = fig.add_subplot(2, 3, 4 + j)
    im = ax.imshow(cond, cmap='inferno',
                   vmin=np.nanpercentile(cond, 1), vmax=np.nanpercentile(cond, 99))
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title(f'condensation — {name}'); ax.axis('off')
plt.tight_layout(); plt.show()
if 'SAVE_DIR' in globals():
    fig.savefig(os.path.join(SAVE_DIR, 'pca_vs_slope3_projection.png'), dpi=120, bbox_inches='tight')

# the matching y-intercept (density) maps for the three axes
fig2, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (name, (s, b, cond, dens)) in zip(axes, methods.items()):
    im = ax.imshow(dens, cmap='viridis',
                   vmin=np.nanpercentile(dens, 1), vmax=np.nanpercentile(dens, 99))
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title(f'density (y-intercept) — {name}'); ax.axis('off')
fig2.suptitle('per-pixel y-intercept b = log10(V_DLS) - slope*log10(1/D*)  (mass density)', fontsize=12)
plt.tight_layout(); plt.show()
if 'SAVE_DIR' in globals():
    fig2.savefig(os.path.join(SAVE_DIR, 'density_yintercept_maps.png'), dpi=120, bbox_inches='tight')

In [ ]:
# ── perpendicular-axis (density) signal-vs-noise test via independent halves ──
# Does the y-intercept (density) axis b = Y - slope*X carry REAL reproducible
# signal, or is it just γ/CV² estimation noise? Split the video into two
# independent halves, estimate the axes from each, and use test-retest agreement:
#   r(half1, half2) across pixels = reliability of a half-length estimate (0 = pure noise);
#   Spearman-Brown → full-length reliability;  variance split → signal-variance fraction.
import numpy as np
from scipy.stats import pearsonr
from utils.gpu_iscors_fit import gpu_fit_maps, compute_density

T = video_proc.shape[0]; half = T // 2
def _XY(seg):
    d, _ = compute_density(seg, min_cv=0.005)
    f = gpu_fit_maps(seg, recon_taus=RECON_TAUS, n_components=1, global_alpha=True,
                     gamma_scale=GAMMA_SCALE, min_cv=0.005, n_steps=400, verbose=False)
    Xh = np.log10(np.clip(1.0 / np.clip(f['gamma'], 1e-6, None), 1e-12, None))
    Yh = np.log10(np.clip(d, 1e-12, None))
    return Xh, Yh
X1, Y1 = _XY(video_proc[:half])
X2, Y2 = _XY(video_proc[half:])

def _reliability(slope, label):
    # the two orthogonal coordinates, each estimated from each independent half
    perp1, perp2 = Y1 - slope * X1, Y2 - slope * X2                    # density (y-intercept)
    alng1 = (X1 + slope * Y1) / (1 + slope ** 2)                       # condensation (along-line)
    alng2 = (X2 + slope * Y2) / (1 + slope ** 2)
    for name, a1, a2 in [('condensation (along)', alng1, alng2),
                         ('density (perp)',       perp1, perp2)]:
        ok = nuc & np.isfinite(a1) & np.isfinite(a2)
        if ok.sum() < 2:
            print(f'  [{label}] {name:20s} skipped — {ok.sum()} px'); continue
        r = pearsonr(a1[ok], a2[ok])[0]
        r_full = 2 * r / (1 + r) if r > -1 else float('nan')          # half → full length
        var_noise_half = np.var(a1[ok] - a2[ok]) / 2                  # σ²(half-length noise)
        var_full = np.var((a1[ok] + a2[ok]) / 2)                      # full-length estimate var
        sig_frac = max(0.0, 1 - (var_noise_half / 2) / (var_full + 1e-12))
        print(f'  [{label}] {name:20s} r(halves)={r:+.3f}  full-len reliability={r_full:+.3f}'
              f'  signal-var frac≈{sig_frac:.2f}')

print('=== independent-halves reliability (nucleus pixels) ===')
print('  r→0 ⇒ axis is pure estimation noise;  r→1 ⇒ reproducible real signal\n')
_reliability(3.0,   'slope-3')
_reliability(s_pca, 'PCA   ')

In [ ]:
# ── reweighting comparison: Gaussian-PDF (typicality) vs half-split (reliability) ──
# Same map (log CV² density), two confidence weights:
#   (a) w_pdf = 2D-Gaussian PDF at the pixel's (log1/D*, logCV²)  → "how TYPICAL is the value"
#   (b) w_rel = small half-split discrepancy                       → "how REPRODUCIBLE is it"
# Core test: among TAIL pixels (extreme values), how does each weight treat the
# REPRODUCIBLE tail (real signal) vs the NOISY tail (noise)?  (needs cells 8 & 9 run first)
import os, numpy as np, matplotlib.pyplot as plt
from scipy.stats import pearsonr

# half-split feature maps (reuse cell-9 results if present, else recompute)
if not all(k in globals() for k in ('X1', 'Y1', 'X2', 'Y2')):
    from utils.gpu_iscors_fit import gpu_fit_maps, compute_density
    T = video_proc.shape[0]; half = T // 2
    def _XY(seg):
        d, _ = compute_density(seg, min_cv=0.005)
        f = gpu_fit_maps(seg, recon_taus=RECON_TAUS, n_components=1, global_alpha=True,
                         gamma_scale=GAMMA_SCALE, min_cv=0.005, n_steps=400, verbose=False)
        return (np.log10(np.clip(1.0 / np.clip(f['gamma'], 1e-6, None), 1e-12, None)),
                np.log10(np.clip(d, 1e-12, None)))
    X1, Y1 = _XY(video_proc[:half]); X2, Y2 = _XY(video_proc[half:])

# the map we reweight = the reliable density channel log CV² (Y from cell 8)
val = Y.copy()
m = (nuc & np.isfinite(X) & np.isfinite(Y)
     & np.isfinite(X1) & np.isfinite(Y1) & np.isfinite(X2) & np.isfinite(Y2))

# (a) Gaussian-PDF (typicality) weight: Mahalanobis distance in the 2D (X,Y) cloud
P = np.vstack([X[m], Y[m]]); mu = P.mean(1); Ci = np.linalg.inv(np.cov(P))
d = np.vstack([(X - mu[0]).ravel(), (Y - mu[1]).ravel()])
maha2 = np.einsum('ij,ij->j', Ci @ d, d).reshape(X.shape)
w_pdf = np.exp(-0.5 * maha2); w_pdf[~nuc] = np.nan          # ∝ Gaussian PDF

# (b) half-split reliability weight: value-blind, only how far the pixel MOVES between halves
disc2 = (X1 - X2) ** 2 + (Y1 - Y2) ** 2                     # per-pixel 2D discrepancy²
med = np.nanmedian(disc2[m])
w_rel = med / (disc2 + med); w_rel[~nuc] = np.nan          # =1 at disc 0, =0.5 at median disc

def _norm(w):
    lo, hi = np.nanpercentile(w[nuc], 1), np.nanpercentile(w[nuc], 99)
    return np.clip((w - lo) / (hi - lo + 1e-12), 0, 1)
w_pdf_n, w_rel_n = _norm(w_pdf), _norm(w_rel)

# ---- core test: tail pixels split into reproducible (real) vs noisy ----
tail = m & (np.abs(val - np.nanmean(val[m])) > 1.5 * np.nanstd(val[m]))    # extreme-value pixels
dmed = np.nanmedian(disc2[tail])
real_tail  = tail & (disc2 <= dmed)            # extreme AND reproducible  → real signal
noise_tail = tail & (disc2 >  dmed)            # extreme AND irreproducible → noise
print('=== weight given to TAIL pixels (|value-mean| > 1.5σ) ===')
print(f'  tail={tail.sum()}  (real {real_tail.sum()} / noise {noise_tail.sum()})')
for label, w in [('Gaussian-PDF (typicality)', w_pdf_n), ('half-split (reliability)', w_rel_n)]:
    wr, wn = np.nanmean(w[real_tail]), np.nanmean(w[noise_tail])
    print(f'  {label:26s} real-tail w={wr:.3f}  noise-tail w={wn:.3f}  ratio={wr/(wn+1e-9):.2f}')
print('  → ratio≈1 ⇒ weight cannot tell real tail from noise; ratio≫1 ⇒ keeps real, kills noise')
repro = -np.log10(disc2 + 1e-9)                # higher = more reproducible (less noisy)
for label, w in [('Gaussian-PDF', w_pdf_n), ('half-split', w_rel_n)]:
    ok = m & np.isfinite(repro) & np.isfinite(w)
    print(f'  corr(weight, reproducibility): {label:14s} r={pearsonr(w[ok], repro[ok])[0]:+.3f}')

# ---- plots ----
fig, ax = plt.subplots(2, 3, figsize=(16, 9))
def _show(a, dd, t, cmap='inferno'):
    im = a.imshow(dd, cmap=cmap, vmin=np.nanpercentile(dd, 1), vmax=np.nanpercentile(dd, 99))
    plt.colorbar(im, ax=a, fraction=0.046); a.set_title(t, fontsize=10); a.axis('off')
_show(ax[0, 0], np.where(nuc, val, np.nan), 'map: log CV² (original)')
_show(ax[0, 1], w_pdf_n, '(a) Gaussian-PDF weight', 'viridis')
_show(ax[0, 2], w_rel_n, '(b) half-split reliability weight', 'viridis')
_show(ax[1, 0], np.where(real_tail, val, np.nan), 'REAL tail (extreme & reproducible)')
_show(ax[1, 1], np.where(nuc, val * w_pdf_n, np.nan), 'map × PDF weight')
_show(ax[1, 2], np.where(nuc, val * w_rel_n, np.nan), 'map × reliability weight')
fig.suptitle('Reweighting: typicality (PDF) vs measured reliability (half-split)', fontsize=13)
plt.tight_layout(); plt.show()
if 'SAVE_DIR' in globals():
    fig.savefig(os.path.join(SAVE_DIR, 'reweight_pdf_vs_reliability.png'), dpi=120, bbox_inches='tight')

In [ ]:
# ── value-blind weight #3: theoretical σ_G (Wiener-Khinchin), no data splitting ──
# half-split is value-blind but per-pixel noisy (only 2 estimates). The analytic ACF
# noise σ_G(τ) ≈ sqrt((2/T)(1+G_norm²)) gives a SMOOTH per-pixel noise scale from all
# T frames at once. Test: does it separate real vs noise tail and track the half-split
# reproducibility WITHOUT ever splitting the data? (needs cell 10 run first)
import numpy as np, matplotlib.pyplot as plt, torch
from scipy.stats import pearsonr
from utils.gpu_iscors_fit import compute_g_norm_torch, _fisher_weights_g0

g_norm, _cm, _gz = compute_g_norm_torch(video_proc, RECON_TAUS, norm='nor1', min_cv=0.005)
g_norm = g_norm.cpu().numpy()
T = video_proc.shape[0]
sigma_g_norm = np.sqrt((2.0 / T) * (1.0 + g_norm ** 2))           # (H,W,K) σ_G/G(0), value-blind
fw = _fisher_weights_g0(torch.tensor([float(t) for t in RECON_TAUS])).cpu().numpy()
sigma_px = (sigma_g_norm * fw.reshape(1, 1, -1)).sum(-1)          # (H,W) per-pixel ACF noise scale
med_s = np.nanmedian((sigma_px ** 2)[nuc])
w_theory = med_s / (sigma_px ** 2 + med_s); w_theory[~nuc] = np.nan
w_theory_n = _norm(w_theory)                                      # reuse _norm() from cell 10

print('=== weight #3: theoretical σ_G (no data split) on the SAME tail test ===')
wr, wn = np.nanmean(w_theory_n[real_tail]), np.nanmean(w_theory_n[noise_tail])
print(f'  theoretical σ_G   real-tail w={wr:.3f}  noise-tail w={wn:.3f}  ratio={wr/(wn+1e-9):.2f}')
ok = m & np.isfinite(repro) & np.isfinite(w_theory_n)
print(f'  corr(weight, half-split reproducibility): r={pearsonr(w_theory_n[ok], repro[ok])[0]:+.3f}')
print('  ↳ INDEPENDENT validation — σ_G is analytic, it never saw the half-split disc².')
print('    high r ⇒ σ_G is a valid SMOOTH, split-free drop-in for the noisy half-split weight.')

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
panels = [(w_rel_n, 'half-split weight (noisy)', 'viridis'),
          (w_theory_n, 'theoretical σ_G weight (smooth)', 'viridis'),
          (np.where(nuc, Y * w_theory_n, np.nan), 'log CV² × σ_G weight', 'inferno')]
for a, (dd, t, cmap) in zip(ax, panels):
    im = a.imshow(dd, cmap=cmap, vmin=np.nanpercentile(dd, 1), vmax=np.nanpercentile(dd, 99))
    plt.colorbar(im, ax=a, fraction=0.046); a.set_title(t, fontsize=10); a.axis('off')
plt.tight_layout(); plt.show()
if 'SAVE_DIR' in globals():
    import os; fig.savefig(os.path.join(SAVE_DIR, 'theoretical_sigma_weight.png'), dpi=120, bbox_inches='tight')

In [ ]:
# ── decisive test: is there a condensation pattern INDEPENDENT of density, spatially? ──
# To separate condensation from density you need a dynamics (1/D*) spatial pattern that is
# (a) NOT just a copy of density and (b) reproducible. Test: within each independent half,
# regress 1/D* (X) on density (Y), take the residual X⊥Y (= dynamics with density removed),
# and check whether THAT residual reproduces across the two halves. (needs cells 8 & 9.)
import numpy as np, matplotlib.pyplot as plt
from scipy.stats import pearsonr

if not all(k in globals() for k in ('X1', 'Y1', 'X2', 'Y2')):
    from utils.gpu_iscors_fit import gpu_fit_maps, compute_density
    Tn = video_proc.shape[0]; hh = Tn // 2
    def _XY(seg):
        d, _ = compute_density(seg, min_cv=0.005)
        f = gpu_fit_maps(seg, recon_taus=RECON_TAUS, n_components=1, global_alpha=True,
                         gamma_scale=GAMMA_SCALE, min_cv=0.005, n_steps=400, verbose=False)
        return (np.log10(np.clip(1 / np.clip(f['gamma'], 1e-6, None), 1e-12, None)),
                np.log10(np.clip(d, 1e-12, None)))
    X1, Y1 = _XY(video_proc[:hh]); X2, Y2 = _XY(video_proc[hh:])

mm = nuc & np.isfinite(X1) & np.isfinite(Y1) & np.isfinite(X2) & np.isfinite(Y2)
def _resid(Xh, Yh):                       # X with its density (Y) component regressed out
    a, b = np.polyfit(Yh[mm], Xh[mm], 1)
    r = Xh - (a * Yh + b); r[~nuc] = np.nan
    return r

rY     = pearsonr(Y1[mm], Y2[mm])[0]                       # density channel reproducibility
rX     = pearsonr(X1[mm], X2[mm])[0]                       # raw dynamics reproducibility
rp1, rp2 = _resid(X1, Y1), _resid(X2, Y2)
rXperp = pearsonr(rp1[mm], rp2[mm])[0]                     # density-removed dynamics: THE test

print('=== can condensation be separated from density SPATIALLY? (nucleus, half-split) ===')
print(f'  density  (logCV² = Y)           reproducibility r = {rY:+.3f}   ← reliable channel')
print(f'  dynamics (1/D*   = X)           reproducibility r = {rX:+.3f}')
print(f'  dynamics WITH density removed   reproducibility r = {rXperp:+.3f}   ← THE answer')
print('  r(X⊥Y)≈0 ⇒ density-independent condensation is NOISE ⇒ NOT separable per-pixel')
print('  r(X⊥Y)>0 ⇒ a reproducible condensation pattern distinct from density DOES exist')

# visualise: density map vs the density-removed dynamics (full data)
resid_full = _resid(X, Y)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
panels = [(np.where(nuc, Y, np.nan), 'density (log CV²)', 'inferno'),
          (np.where(nuc, X, np.nan), 'dynamics (log 1/D*)', 'inferno'),
          (resid_full, 'dynamics ⊥ density  (the separable part)', 'coolwarm')]
for a, (dd, t, cmap) in zip(ax, panels):
    im = a.imshow(dd, cmap=cmap, vmin=np.nanpercentile(dd, 2), vmax=np.nanpercentile(dd, 98))
    plt.colorbar(im, ax=a, fraction=0.046); a.set_title(t, fontsize=10); a.axis('off')
fig.suptitle(f'Separability: density-removed dynamics reproduces at r={rXperp:+.3f}', fontsize=12)
plt.tight_layout(); plt.show()
if 'SAVE_DIR' in globals():
    import os; fig.savefig(os.path.join(SAVE_DIR, 'separability_test.png'), dpi=120, bbox_inches='tight')

In [ ]:
# ── follow-up: is the separable signal in BULK chromatin, or only the nucleoli? ──
# r(X⊥Y)=0.41 had structure concentrated at the nucleoli — which are big coherent blobs
# (inflate pixel-wise r) and are not chromatin condensation anyway. Decisive question:
# exclude the nucleoli (defined by DENSITY only, so no circularity) and re-measure whether
# the density-removed dynamics still reproduces in the bulk chromatin. (needs cells 8 & 9.)
import numpy as np, matplotlib.pyplot as plt
from scipy.stats import pearsonr
from scipy.ndimage import binary_dilation

# nucleoli = lowest-density pixels (independent of the X⊥Y residual → not circular)
ythr = np.nanpercentile(Y[nuc], 12)
nucleoli = binary_dilation(nuc & (Y < ythr), iterations=2) & nuc
bulk = nuc & ~nucleoli

def _sep_r(region):
    mr = region & np.isfinite(X1) & np.isfinite(Y1) & np.isfinite(X2) & np.isfinite(Y2)
    if mr.sum() < 10:
        return float('nan'), int(mr.sum()), None
    a1, b1 = np.polyfit(Y1[mr], X1[mr], 1); r1 = X1 - (a1 * Y1 + b1)
    a2, b2 = np.polyfit(Y2[mr], X2[mr], 1); r2 = X2 - (a2 * Y2 + b2)
    return pearsonr(r1[mr], r2[mr])[0], int(mr.sum()), (r1 + r2) / 2

r_all,  n0, _      = _sep_r(nuc)
r_nuc,  n1, _      = _sep_r(nucleoli)
r_bulk, n2, res_b  = _sep_r(bulk)
print('=== where does the separable (density-removed dynamics) signal live? ===')
print(f'  whole nucleus    r(X⊥Y) = {r_all:+.3f}  (n={n0})')
print(f'  nucleoli only    r(X⊥Y) = {r_nuc:+.3f}  (n={n1})')
print(f'  BULK chromatin   r(X⊥Y) = {r_bulk:+.3f}  (n={n2})  ← the one that matters')
print('  bulk≈0 ⇒ separable signal is only the nucleoli (distinct compartments), not a chromatin map')
print('  bulk>0 ⇒ a genuine density-independent condensation pattern within the chromatin')

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].imshow(np.where(nuc, Y, np.nan), cmap='inferno',
             vmin=np.nanpercentile(Y[nuc], 2), vmax=np.nanpercentile(Y[nuc], 98))
ax[0].contour(nucleoli, colors='cyan', linewidths=0.6)
ax[0].set_title('density + excluded nucleoli (cyan)'); ax[0].axis('off')
resmap = np.full_like(Y, np.nan);
if res_b is not None: resmap[bulk] = res_b[bulk]
ax[1].imshow(resmap, cmap='coolwarm',
             vmin=np.nanpercentile(resmap, 2), vmax=np.nanpercentile(resmap, 98))
ax[1].set_title(f'bulk dynamics ⊥ density  (r={r_bulk:+.3f})'); ax[1].axis('off')
ax[2].axis('off')
ax[2].text(0.0, 0.5, f'whole nucleus  r={r_all:+.3f}\nnucleoli only  r={r_nuc:+.3f}\n'
                     f'BULK chromatin r={r_bulk:+.3f}', fontsize=13, va='center', family='monospace')
plt.tight_layout(); plt.show()
if 'SAVE_DIR' in globals():
    import os; fig.savefig(os.path.join(SAVE_DIR, 'separability_bulk_vs_nucleoli.png'), dpi=120, bbox_inches='tight')

In [ ]:
# ── most-REPRODUCIBLE axis, done right: reliability-maximising (generalized eigenproblem) ──
# The previous version maximised signal VARIANCE (ordinary eig of the cross-half covariance),
# which pulls in the noisy 1/D* axis and UNDERPERFORMS. The reliability of a direction w is
#   λ(w) = wᵀ C_sig w / wᵀ C_tot w   (reproducible variance / total variance),
# maximised by the GENERALIZED eigenproblem  C_sig w = λ C_tot w. Its top eigenvalue IS the
# best achievable reliability and its eigenvector is the true most-reproducible axis.
# (needs cells 8 & 9.)
import numpy as np, matplotlib.pyplot as plt
from scipy.stats import pearsonr
from scipy.linalg import eigh as geigh

mm = nuc & np.isfinite(X1) & np.isfinite(Y1) & np.isfinite(X2) & np.isfinite(Y2)
mX, mY = X[mm].mean(), Y[mm].mean(); sX, sY = X[mm].std(), Y[mm].std()
f1 = np.vstack([(X1[mm] - mX) / sX, (Y1[mm] - mY) / sY])
f2 = np.vstack([(X2[mm] - mX) / sX, (Y2[mm] - mY) / sY])
Csig = (f1 @ f2.T) / f1.shape[1]; Csig = (Csig + Csig.T) / 2     # signal cov (noise cancels)
Ctot = np.cov(np.hstack([f1, f2]))                              # total cov (signal + noise)

_, Vv = np.linalg.eigh(Csig); v_var = Vv[:, 1]                  # variance-max axis (flawed)
gval, Gv = geigh(Csig, Ctot); v_rel = Gv[:, -1]                 # reliability-max axis (correct)
slope_rel = (sY * v_rel[1]) / (sX * v_rel[0] + 1e-12)

def _proj(vec, Xs, Ys): return vec[0] * (Xs - mX) / sX + vec[1] * (Ys - mY) / sY
def _rel(vec): return pearsonr(_proj(vec, X1, Y1)[mm], _proj(vec, X2, Y2)[mm])[0]
a, b = np.polyfit(Y1[mm], X1[mm], 1); rh1 = X1 - (a * Y1 + b)   # X⊥Y residual, per half
a, b = np.polyfit(Y2[mm], X2[mm], 1); rh2 = X2 - (a * Y2 + b)
r_xperp = pearsonr(rh1[mm], rh2[mm])[0]
af, bf = np.polyfit(Y[mm], X[mm], 1); resid_full = X - (af * Y + bf); resid_full[~nuc] = np.nan

print('=== reliability r(halves) of candidate condensation axes ===')
print(f'  reliability-MAX axis (generalized)  = {_rel(v_rel):+.3f}   slope={slope_rel:+.2f}   ← correct')
print(f'  (theoretical best λ                 = {gval[-1]:+.3f})')
print(f'  variance-MAX axis (ordinary eig)    = {_rel(v_var):+.3f}   ← previous, flawed')
print(f'  X⊥Y residual (density removed)      = {r_xperp:+.3f}')
print(f'  raw density logCV²                  = {pearsonr(Y1[mm], Y2[mm])[0]:+.3f}')

cond = _proj(v_rel, X, Y); cond[~nuc] = np.nan
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
panels = [(cond, f'condensation = reliability-max axis (slope {slope_rel:.2f})', 'inferno'),
          (resid_full, 'X⊥Y residual (previous best)', 'coolwarm'),
          (np.where(nuc, Y, np.nan), 'raw log CV² (density ref)', 'inferno')]
for a_, (dd, t, cmap) in zip(ax, panels):
    im = a_.imshow(dd, cmap=cmap, vmin=np.nanpercentile(dd, 2), vmax=np.nanpercentile(dd, 98))
    plt.colorbar(im, ax=a_, fraction=0.046); a_.set_title(t, fontsize=10); a_.axis('off')
plt.tight_layout(); plt.show()
if 'SAVE_DIR' in globals():
    import os; fig.savefig(os.path.join(SAVE_DIR, 'reliability_max_axis.png'), dpi=120, bbox_inches='tight')

In [ ]:
# ── pixel train/test cross-validation of the reliability-max axis ──
# The axis is fit AND scored on the same pixels → mildly optimistic (and λ>1 flags it).
# Honest test: fit v_rel on a random TRAIN subset of pixels, score its cross-half
# reliability on the held-out TEST pixels. Repeat K folds. (needs cells 8 & 9.)
import numpy as np
from scipy.stats import pearsonr
from scipy.linalg import eigh as geigh

mm = nuc & np.isfinite(X1) & np.isfinite(Y1) & np.isfinite(X2) & np.isfinite(Y2)
idx = np.flatnonzero(mm.ravel())
X1f, Y1f, X2f, Y2f = X1.ravel(), Y1.ravel(), X2.ravel(), Y2.ravel()
mXa, mYa = X[mm].mean(), Y[mm].mean(); sXa, sYa = X[mm].std(), Y[mm].std()

def _fit_axis(ix):                                   # generalized eigvec on a pixel subset
    f1 = np.vstack([(X1f[ix] - mXa) / sXa, (Y1f[ix] - mYa) / sYa])
    f2 = np.vstack([(X2f[ix] - mXa) / sXa, (Y2f[ix] - mYa) / sYa])
    Cs = (f1 @ f2.T) / f1.shape[1]; Cs = (Cs + Cs.T) / 2
    Ct = np.cov(np.hstack([f1, f2]))
    return geigh(Cs, Ct)[1][:, -1]
def _score(vec, ix):                                 # cross-half reliability on a subset
    p1 = vec[0] * (X1f[ix] - mXa) / sXa + vec[1] * (Y1f[ix] - mYa) / sYa
    p2 = vec[0] * (X2f[ix] - mXa) / sXa + vec[1] * (Y2f[ix] - mYa) / sYa
    return pearsonr(p1, p2)[0]

rng = np.random.default_rng(0)
ins, outs = [], []
for k in range(5):
    perm = rng.permutation(idx); cut = len(perm) // 2
    tr, te = perm[:cut], perm[cut:]
    v = _fit_axis(tr)
    ins.append(_score(v, tr)); outs.append(_score(v, te))
ins, outs = np.array(ins), np.array(outs)
print('=== pixel train/test cross-validation (reliability-max axis, 5 folds) ===')
print(f'  in-sample (train) r  = {ins.mean():.3f} ± {ins.std():.3f}')
print(f'  OUT-of-sample (test) r = {outs.mean():.3f} ± {outs.std():.3f}   ← honest value')
print(f'  optimism gap (train-test) = {ins.mean()-outs.mean():+.3f}')
print('  gap≈0 ⇒ the axis is NOT overfit; the test r is the trustworthy reliability.')

In [ ]:
# ── (1) reliability landscape over θ  +  (2) VALID drift test (block time-gap) ──
# (1) Scan cross-half reliability over all directions θ on the CONTIGUOUS split (the valid one):
#     #peaks = #reproducible axes. (2) Interleaved odd/even is INVALID here — frame interval ≪
#     intensity correlation time ⇒ odd/even are not independent ⇒ r→1 trivially (shown for
#     contrast, NOT a drift test). The valid drift test projects 4 contiguous quarters onto the
#     reproducible axis and checks whether reliability decays with their time gap. (cells 8 & 9.)
import numpy as np, matplotlib.pyplot as plt
from scipy.stats import pearsonr
from scipy.linalg import eigh as geigh
from utils.gpu_iscors_fit import gpu_fit_maps, compute_density

def _XY(seg):
    d, _ = compute_density(seg, min_cv=0.005)
    f = gpu_fit_maps(seg, recon_taus=RECON_TAUS, n_components=1, global_alpha=True,
                     gamma_scale=GAMMA_SCALE, min_cv=0.005, n_steps=400, verbose=False)
    return (np.log10(np.clip(1 / np.clip(f['gamma'], 1e-6, None), 1e-12, None)),
            np.log10(np.clip(d, 1e-12, None)))

mX, mY = X[nuc].mean(), Y[nuc].mean(); sX, sY = X[nuc].std(), Y[nuc].std()
th = np.linspace(0, np.pi, 181)
def _land(Xp, Yp, Xq, Yq, msk):
    out = []
    for t in th:
        c, s = np.cos(t), np.sin(t)
        out.append(pearsonr(c * (Xp[msk] - mX) / sX + s * (Yp[msk] - mY) / sY,
                            c * (Xq[msk] - mX) / sX + s * (Yq[msk] - mY) / sY)[0])
    return np.array(out)

Xa, Ya = _XY(video_proc[0::2]); Xb, Yb = _XY(video_proc[1::2])   # interleaved (shown for contrast)
mC = nuc & np.isfinite(X1) & np.isfinite(Y1) & np.isfinite(X2) & np.isfinite(Y2)
mI = nuc & np.isfinite(Xa) & np.isfinite(Ya) & np.isfinite(Xb) & np.isfinite(Yb)
rC, rI = _land(X1, Y1, X2, Y2, mC), _land(Xa, Ya, Xb, Yb, mI)
def _npk(r): return int(sum(1 for i in range(1, len(r) - 1)
                            if r[i] > r[i - 1] and r[i] > r[i + 1] and r[i] > 0.5 * r.max()))
print('=== reliability landscape (CONTIGUOUS split = the valid one) ===')
print(f'  contiguous : peak r={rC.max():.3f} @ {np.degrees(th[rC.argmax()]):.0f}°, peaks={_npk(rC)}  ← #reproducible axes')
print(f'  interleaved: ~{rI.mean():.2f} at every angle → INVALID (odd/even not independent:')
print(f'               frame interval ≪ correlation time ⇒ r→1, NOT a drift test)')

# valid drift test: reproducible-axis reliability vs time-gap between contiguous quarters
f1 = np.vstack([(X1[mC] - mX) / sX, (Y1[mC] - mY) / sY]); f2 = np.vstack([(X2[mC] - mX) / sX, (Y2[mC] - mY) / sY])
Cs = (f1 @ f2.T) / f1.shape[1]; Cs = (Cs + Cs.T) / 2; v = geigh(Cs, np.cov(np.hstack([f1, f2])))[1][:, -1]
nq = 4; L = video_proc.shape[0] // nq
P = []
for i in range(nq):
    Xq, Yq = _XY(video_proc[i * L:(i + 1) * L]); P.append(v[0] * (Xq - mX) / sX + v[1] * (Yq - mY) / sY)
mq = nuc.copy()
for p in P: mq &= np.isfinite(p)
print('\n=== drift test: reproducible-axis reliability vs block time-gap (4 contiguous quarters) ===')
for gap in (1, 2, 3):
    rs = [pearsonr(P[i][mq], P[i + gap][mq])[0] for i in range(nq - gap)]
    print(f'  gap={gap} block(s): r={np.mean(rs):+.3f}')
print('  flat across gap ⇒ stationary (no slow drift);  decreasing ⇒ slow drift/non-stationarity')

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(np.degrees(th), rC, 'b-',  label=f'contiguous (valid)   max={rC.max():.2f}')
ax.plot(np.degrees(th), rI, 'g--', label=f'interleaved (invalid) ~{rI.mean():.2f}')
ax.set_xlabel('projection angle θ (deg)'); ax.set_ylabel('cross-half reliability r')
ax.set_title('#peaks (contiguous) = #reproducible axes; interleaved invalid here')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
if 'SAVE_DIR' in globals():
    import os; fig.savefig(os.path.join(SAVE_DIR, 'reliability_landscape.png'), dpi=120, bbox_inches='tight')

In [ ]:
# ── does the single peak survive at higher SNR? landscape at 2000 vs 5000 frames ──
# More frames lowers estimation noise (∝1/T) so the WHOLE landscape lifts. The question is
# whether the SHAPE changes: if a 2nd peak (a 2nd reproducible axis) was buried in noise at
# 2000, it should emerge at 5000. Same shape, just lifted ⇒ genuinely one axis.
import numpy as np, matplotlib.pyplot as plt
from scipy.stats import pearsonr
from utils.gpu_iscors_fit import gpu_fit_maps, compute_density

def _XY(seg):
    d, _ = compute_density(seg, min_cv=0.005)
    f = gpu_fit_maps(seg, recon_taus=RECON_TAUS, n_components=1, global_alpha=True,
                     gamma_scale=GAMMA_SCALE, min_cv=0.005, n_steps=400, verbose=False)
    return (np.log10(np.clip(1 / np.clip(f['gamma'], 1e-6, None), 1e-12, None)),
            np.log10(np.clip(d, 1e-12, None)))

mX, mY = X[nuc].mean(), Y[nuc].mean(); sX, sY = X[nuc].std(), Y[nuc].std()
th = np.linspace(0, np.pi, 181)
def _land(H1, H2, msk):
    (Xp, Yp), (Xq, Yq) = H1, H2; out = []
    for t in th:
        c, s = np.cos(t), np.sin(t)
        out.append(pearsonr(c * (Xp[msk] - mX) / sX + s * (Yp[msk] - mY) / sY,
                            c * (Xq[msk] - mX) / sX + s * (Yq[msk] - mY) / sY)[0])
    return np.array(out)

fig, ax = plt.subplots(figsize=(7.5, 5))
for nfr, col in [(2000, 'b'), (5000, 'r')]:
    raw = load_video(ZIP_PATH, VIDEO_FNAME, nfr, BIN_FACTOR); vp = preprocess(raw); h = nfr // 2
    H1, H2 = _XY(vp[:h]), _XY(vp[h:])
    msk = nuc & np.isfinite(H1[0]) & np.isfinite(H1[1]) & np.isfinite(H2[0]) & np.isfinite(H2[1])
    r = _land(H1, H2, msk)
    npk = sum(1 for i in range(1, len(r) - 1) if r[i] > r[i - 1] and r[i] > r[i + 1] and r[i] > 0.5 * r.max())
    ax.plot(np.degrees(th), r, col,
            label=f'{nfr} frames: peak {r.max():.2f}@{np.degrees(th[r.argmax()]):.0f}°, #peaks={npk}, min={r.min():.2f}')
    print(f'{nfr} frames: peak r={r.max():.3f}, #peaks={npk}, min r={r.min():.3f}')
ax.set_xlabel('projection angle θ (deg)'); ax.set_ylabel('cross-half reliability r')
ax.set_title('2nd peak at higher SNR? landscape 2000 vs 5000 frames')
ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
if 'SAVE_DIR' in globals():
    import os; fig.savefig(os.path.join(SAVE_DIR, 'landscape_2000_vs_5000.png'), dpi=120, bbox_inches='tight')

In [ ]:
# ── how many reproducible axes does the FULL ACF contain? (full-τ GEVD) ──
# (γ, CV²) is only 2 numbers; the per-pixel ACF G(τ) has K lags. Use the FULL K-vector as the
# feature and solve the generalized eigenproblem on cross-half vs total covariance. The number
# of generalized eigenvalues clearly above a shuffled-noise floor = number of reproducible axes.
import numpy as np, matplotlib.pyplot as plt
from scipy.linalg import eigh as geigh
from utils.gpu_iscors_fit import compute_g_norm_torch

T = video_proc.shape[0]; h = T // 2
def _G(seg):
    g, _cm, _gz = compute_g_norm_torch(seg, RECON_TAUS, norm='nor1', min_cv=0.005)
    return g.cpu().numpy()                       # (H,W,K) ACF shape per pixel
G1, G2 = _G(video_proc[:h]), _G(video_proc[h:])
K = G1.shape[-1]
m = nuc.copy()
for k in range(K): m &= np.isfinite(G1[..., k]) & np.isfinite(G2[..., k])
F1, F2 = G1[m], G2[m]                            # (N,K)
mu = (F1.mean(0) + F2.mean(0)) / 2; sd = (F1.std(0) + F2.std(0)) / 2 + 1e-9
F1, F2 = (F1 - mu) / sd, (F2 - mu) / sd          # standardise each lag
Ccross = (F1.T @ F2) / F1.shape[0]; Csig = (Ccross + Ccross.T) / 2
Ctot = np.cov(np.vstack([F1, F2]).T)
gv = geigh(Csig, Ctot, eigvals_only=True)[::-1]  # descending reproducible-fraction per component

rng = np.random.default_rng(0); perm = rng.permutation(F2.shape[0])   # break pixel correspondence
Csh = (F1.T @ F2[perm]) / F1.shape[0]; Csh = (Csh + Csh.T) / 2
floor = float(np.abs(geigh(Csh, Ctot, eigvals_only=True)).max())
n_axes = int((gv > floor).sum())
print(f'full-τ GEVD on K={K} ACF lags  (N={F1.shape[0]} pixels):')
print('  reproducible eigenvalues (desc):', np.round(gv, 3))
print(f'  noise floor (pixel-shuffled)   : {floor:.3f}')
print(f'  → # reproducible axes above noise = {n_axes}')
print('  1 ⇒ the full ACF still holds only ONE reproducible spatial axis;')
print('  ≥2 ⇒ extra reproducible structure that (γ,CV²) threw away.')

plt.figure(figsize=(7, 4))
plt.plot(range(1, K + 1), gv, 'o-', label='reproducible λ (real)')
plt.axhline(floor, color='r', ls='--', label=f'noise floor {floor:.2f}')
plt.xlabel('component'); plt.ylabel('cross-half reproducible fraction λ')
plt.title(f'Full-τ GEVD: {n_axes} reproducible axis/axes above noise (K={K})')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
if 'SAVE_DIR' in globals():
    import os; plt.savefig(os.path.join(SAVE_DIR, 'full_tau_gevd.png'), dpi=120, bbox_inches='tight')

In [ ]:
# ── classify the GEVD axes: spatial maps + reliability vs frames (biology/fixed-pattern/noise) ──
# Full-τ GEVD found ~4 reproducible axes. But reproducible ≠ biological (fixed-pattern noise is
# also reproducible). Two diagnostics: (a) each axis's SPATIAL map (biology = nucleoli/foci;
# fixed-pattern = grid); (b) reliability vs frame count — RISING ⇒ real signal denoising,
# FLAT-HIGH ⇒ fixed-pattern artifact, ≈0 ⇒ noise. Axes are fixed from the cleanest (5000) data.
import numpy as np, matplotlib.pyplot as plt
from scipy.linalg import eigh as geigh
from scipy.stats import pearsonr
from utils.gpu_iscors_fit import compute_g_norm_torch

def _halfG(seg):
    g, _c, _z = compute_g_norm_torch(seg, RECON_TAUS, norm='nor1', min_cv=0.005); return g.cpu().numpy()
def _feats(nfr):
    raw = load_video(ZIP_PATH, VIDEO_FNAME, nfr, BIN_FACTOR); vp = preprocess(raw); h = nfr // 2
    return _halfG(vp[:h]), _halfG(vp[h:])

G1, G2 = _feats(5000); K = G1.shape[-1]                         # fix axes from cleanest data
m = nuc.copy()
for k in range(K): m &= np.isfinite(G1[..., k]) & np.isfinite(G2[..., k])
F1, F2 = G1[m], G2[m]
mu = (F1.mean(0) + F2.mean(0)) / 2; sd = (F1.std(0) + F2.std(0)) / 2 + 1e-9
_std = lambda F: (F - mu) / sd
A, B = _std(F1), _std(F2)
Cs = (A.T @ B) / A.shape[0]; Cs = (Cs + Cs.T) / 2; Ct = np.cov(np.vstack([A, B]).T)
val, W = geigh(Cs, Ct); order = np.argsort(val)[::-1]; val, W = val[order], W[:, order]
ntop = 4

print('reliability of each fixed GEVD axis vs frame count:')
print(f'{"frames":>7} ' + ' '.join(f'ax{j+1:>5}' for j in range(ntop)))
for nfr in (1000, 2000, 5000):
    g1, g2 = _feats(nfr)
    mm = nuc.copy()
    for k in range(K): mm &= np.isfinite(g1[..., k]) & np.isfinite(g2[..., k])
    P1, P2 = _std(g1[mm]) @ W, _std(g2[mm]) @ W
    rs = [pearsonr(P1[:, j], P2[:, j])[0] for j in range(ntop)]
    print(f'{nfr:>7} ' + ' '.join(f'{r:>6.2f}' for r in rs))
print('  RISING ⇒ real signal (denoising);  FLAT-HIGH ⇒ fixed-pattern artifact;  ≈0 ⇒ noise')

proj = ((_std(F1) + _std(F2)) / 2) @ W                          # axis maps from 5000 data
fig, ax = plt.subplots(1, ntop, figsize=(4 * ntop, 4))
for j in range(ntop):
    mp = np.full(nuc.shape, np.nan); mp[m] = proj[:, j]
    im = ax[j].imshow(mp, cmap='coolwarm', vmin=np.nanpercentile(mp, 2), vmax=np.nanpercentile(mp, 98))
    ax[j].set_title(f'axis {j+1} (λ={val[j]:.2f})'); ax[j].axis('off'); plt.colorbar(im, ax=ax[j], fraction=0.046)
fig.suptitle('Top reproducible GEVD axes — biology (nucleoli/foci) vs grid (fixed-pattern)?')
plt.tight_layout(); plt.show()
if 'SAVE_DIR' in globals():
    import os; fig.savefig(os.path.join(SAVE_DIR, 'gevd_axis_maps.png'), dpi=120, bbox_inches='tight')

In [ ]:
# ── is K=10 enough? extend τ to longer lags and recount reproducible axes (at 5000 frames) ──
# If reproducible dynamics live beyond τ=128 (~25 ms), adding longer lags should raise the axis
# count above 4. If the count stays ~4, the ACF sampling is NOT the bottleneck — you've extracted
# all the reproducible shape DOF the acquisition holds.
import numpy as np
from scipy.linalg import eigh as geigh
from utils.gpu_iscors_fit import compute_g_norm_torch

raw = load_video(ZIP_PATH, VIDEO_FNAME, 5000, BIN_FACTOR); vp = preprocess(raw); h = vp.shape[0] // 2

def _naxes(taus):
    def _G(seg):
        g, _c, _z = compute_g_norm_torch(seg, taus, norm='nor1', min_cv=0.005); return g.cpu().numpy()
    G1, G2 = _G(vp[:h]), _G(vp[h:]); K = len(taus)
    m = nuc.copy()
    for k in range(K): m &= np.isfinite(G1[..., k]) & np.isfinite(G2[..., k])
    F1, F2 = G1[m], G2[m]
    mu = (F1.mean(0) + F2.mean(0)) / 2; sd = (F1.std(0) + F2.std(0)) / 2 + 1e-9
    A, B = (F1 - mu) / sd, (F2 - mu) / sd
    Cs = (A.T @ B) / A.shape[0]; Cs = (Cs + Cs.T) / 2; Ct = np.cov(np.vstack([A, B]).T)
    gv = geigh(Cs, Ct, eigvals_only=True)[::-1]
    rng = np.random.default_rng(0); pm = rng.permutation(B.shape[0])
    Csh = (A.T @ B[pm]) / A.shape[0]; Csh = (Csh + Csh.T) / 2
    floor = float(np.abs(geigh(Csh, Ct, eigvals_only=True)).max())
    return gv, floor, int((gv > floor).sum())

for name, taus in [('K=10 (max τ=128)',  (1, 2, 4, 8, 16, 32, 48, 64, 96, 128)),
                   ('K=14 (max τ=512)',  (1, 2, 4, 8, 16, 32, 48, 64, 96, 128, 192, 256, 384, 512))]:
    gv, floor, n = _naxes(taus)
    print(f'{name:>18}: #axes above noise = {n}   floor={floor:.3f}')
    print(f'                    eigvals = {np.round(gv[:8], 3)}')
print('  same count ⇒ τ-sampling is NOT the bottleneck (acquisition-limited, not lag-limited).')

In [ ]:
# ── Gradio front-end (layout preview) ────────────────────────────────────
import gradio as gr
def _run(file, bin_factor, n_frames):
    if file is None: raise gr.Error('請上傳 .tif（或含 tif 的 .zip）')
    fname = VIDEO_FNAME if str(file).lower().endswith('.zip') else None
    raw = load_video(file, fname, int(n_frames), int(bin_factor))
    res = analyze(preprocess(raw))
    stats = (f'cell pixels      : {100*res["cell"].mean():.1f}%\n'
             f'apparent global α: {res["alpha_global"]:.3f}  (α=1 ⇒ normal)\n'
             f'γ  median(cell)  : {np.nanmedian(res["gamma"][res["cell"]]):.3f}\n'
             f'density median   : {np.nanmedian(res["density"][res["cell"]]):.4f}  (CV²)')
    return (make_fig(res['gamma'],'magma','γ (diffusion rate)'),
            make_fig(res['density'],'viridis','density  G(0)=CV²'),
            make_fig(res['condensation'],'inferno','Condensation (slope-3 proj)'), stats)

with gr.Blocks(title='iSCORS classical (γ + α + density)') as demo:
    gr.Markdown('## iSCORS classical deliverable — GPU fit, no ML\n'
                'γ map + apparent global α + density G(0)=CV². Seconds, no training, no checkpoint.')
    with gr.Row():
        with gr.Column(scale=1):
            f_in = gr.File(label='影片 (.tif 或含 tif 的 .zip)', file_types=['.tif','.tiff','.zip'], type='filepath')
            b_in = gr.Slider(label='BIN_FACTOR', minimum=1, maximum=8, step=1, value=2)
            n_in = gr.Number(label='N_FRAMES', value=2000, precision=0)
            run  = gr.Button('分析', variant='primary')
        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.Tab('γ 擴散速率'):       g_out = gr.Plot()
                with gr.Tab('密度 G(0)=CV²'):    d_out = gr.Plot()
                with gr.Tab('Condensation'):    c_out = gr.Plot()
            s_out = gr.Textbox(label='統計摘要', lines=5)
    run.click(_run, inputs=[f_in, b_in, n_in], outputs=[g_out, d_out, c_out, s_out])

demo.launch(share=True)
